---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [13]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\georg\OneDrive\Dokument\Claude\Projects\Cursul Inginerie Ai\echochamber-project-team3
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [14]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[b1tvchannel] De la 13:00 încolo, Ioana Constantin murea de frică ca nu cumva Papahagi să dea 
[georgesimionoficial] SIMION A MULȚUMIT CASEI ALBE PENTRU ANULAREA VIZELOR SUA PENTRU ROMÂNI. Trădător
[RecorderRomania] Nu blocarea cazurilor este cea mai mizerabilă acțiune a procurorilor, ci execuți


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [15]:
# modifica dupa preferinte

AXA_1 = "media_distrust"
AXA_2 = "religious_frame"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [16]:
AXA_1_DEFINITION = """
media_distrust măsoară dacă textul exprimă neîncredere în presă, jurnaliști,
televiziuni sau media mainstream.
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
religious_frame măsoară dacă textul folosește limbaj religios pentru a interpreta politica.
0 = absent
1 = prezent
2 = dominant
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [17]:
MINI_PROMPT = f"""
Ești un analist riguros de discurs politic românesc. Lucrezi pentru cercetare academică. Adnotezi comentarii YouTube cu detașare și precizie, fără a lua poziție politică.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un analist riguros de discurs politic românesc. Lucrezi pentru cercetare academică. Adnotezi comentarii YouTube cu detașare și precizie, fără a lua poziție politică.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. media_distrust
2. religious_frame
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
media_distrust = 0 / 1 / 2
religious_frame = 0 / 1 / 2
DEFINIȚII:

media_distrust măsoară dacă textul exprimă neîncredere în presă, jurnaliști,
televiziuni sau media mainstream.
0 = absent
1 = prezent
2 = dominant


religious_frame măsoară dacă textul folosește limbaj religios pentru a interpreta politica.
0 = absent
1 = prezent
2 = dominant

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="n

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [18]:
TESTS = corpus.sample(5, random_state=42)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
145,yt_BfvZ8QcVBKc_Ugyog0iMEqAX5zIQb4R4AaABAg,turcescu111,Harpalete- Sângerete și transfuzia din lumea lui,"Da, si eu cred ca serviciile ucrainene au fost..."
334,yt_qkGhsJFft00_UgzJHKCoGkwtTJ3uOBh4AaABAg,digi24hd56,În fața ta cu Emil Hurezeanu: „Ar fi un coșmar...,Ce mă supără pe mine oamenii ăștia care sunt a...
175,yt_KqrUotq1Obs_Ugy6tYmdfH0T2THArnB4AaABAg,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pace și prosperitate ( 28.10...,Bunul Dumnezeu să îl protejeze pe președintele...
369,yt_vkP6FdP9iX0_Ugwj3HojTt6JikJVtjd4AaABAg,turcescu111,Orientul Mijlociu în flăcări,Nicușor merge pe lângă covor pentru că nu are ...
416,yt_Sj4fQKlMOro_UgyNWIwNDtUeTTCrLkl4AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Trebuie susținută aceasta femeie!!!!! Acesta a...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [19]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [10]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [20]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 37.809400808s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash-lite', 'location': 'g

In [21]:
## Mini-tipologie bazată pe cele 2 axe

# Parsăm JSON-urile din results și aplicăm tipologia
parsed = []
for r in results:
    raw = r["model_output"].strip()
    # eliminăm eventualele blocuri markdown ```json ... ```
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"```$", "", raw.strip())
    try:
        obj = json.loads(raw)
    except Exception:
        obj = {"target": "?", "stance": "?", "tone": "?", "media_distrust": -1, "religious_frame": -1}
    obj["text_preview"] = r["text"][:60]
    parsed.append(obj)

# Reguli tipologie
def tipologie(md, rf):
    if md == 0 and rf == 0:
        return "T0_neutru_analitic"
    elif md >= 1 and rf == 0:
        return "T1_sceptic_mediatic"
    elif md == 0 and rf >= 1:
        return "T2_afectiv_religios"
    elif md >= 1 and rf >= 1:
        return "T3_suveranist_religios"
    return "T_necunoscut"

for p in parsed:
    md = p.get("media_distrust", 0)
    rf = p.get("religious_frame", 0)
    tip = tipologie(md, rf)
    print(f"[{tip}] target={p.get('target','?')} | {p['text_preview']}")


## Pasul 6 — Interpretare scurtă

Am ales axele **media_distrust** și **religious_frame** deoarece corespund profilului S6 (intelectual-critic, sceptic): primul tip de discurs pe care îl identific în corpusul meu este scepticismul față de media mainstream, al doilea este recursul la cadre religioase în argumentul politic  două tendințe clare în comentariile de pe RecorderRomania și EuropaLibera.

Modelul a returnat JSON valid în majoritatea cazurilor. Problema principală a fost că pentru comentarii ambigue sau scurte, modelul a setat toate axele pe 0 fără justificare, ceea ce sugerează că definițiile din prompt nu sunt suficient de precise pentru cazurile limită.

Aș adăuga în prompt 2-3 exemple concrete (few-shot) pentru fiecare axă, în special pentru cazul în care media_distrust este implicit (ton acuzator fără a numi direct presa).